In [1]:
# =========================================================
# 01. 라이브러리 불러오기
# =========================================================

import pandas as pd
import ast


# =========================================================
# 02. 실행 조건 / 파일 경로 설정
# =========================================================

INPUT_PATH = "../../../../data/preprocessed/steam_indie_games_graded.csv"
SAVE_PATH = "../../../../data/preprocessed/tableau_genre_long.csv"

TARGET_GENRES = [
    "Action",
    "Adventure",
    "Casual",
    "Racing",
    "RPG",
    "Simulation",
    "Sports",
    "Strategy"
]

GRADE_LABEL = {
    "high": "높음",
    "mid": "중간",
    "low": "낮음"
}


# =========================================================
# 03. 데이터 불러오기
# =========================================================

games = pd.read_csv(INPUT_PATH)

print("원본 데이터 크기:", games.shape)
print(games.columns.tolist())


# =========================================================
# 04. genres 컬럼을 실제 리스트로 변환하는 함수
# =========================================================

def parse_list(x):
    """
    목적:
        문자열로 저장된 리스트를 실제 파이썬 list로 바꾼다.

    예시 입력:
        "['Action', 'Adventure']"

    예시 출력:
        ['Action', 'Adventure']

    변환 실패 시:
        빈 리스트 [] 반환
    """

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        value = ast.literal_eval(str(x))

        if isinstance(value, list):
            return value
        else:
            return []

    except:
        return []


# =========================================================
# 05. genres 리스트 변환
# =========================================================

games["genres_list"] = games["genres"].apply(parse_list)

print("변환 확인:")
display(games[["appid", "name", "genres", "genres_list"]].head())


# =========================================================
# 06. 분석 대상 장르만 남기기
# =========================================================

games["genres_filtered"] = games["genres_list"].apply(
    lambda genre_list: [
        genre for genre in genre_list
        if genre in TARGET_GENRES
    ]
)

games_with_genre = games[
    games["genres_filtered"].map(len) > 0
].copy()

print("대상 장르가 있는 원본 게임 수:", len(games_with_genre))


# =========================================================
# 07. explode로 장르 1개당 1행 만들기
# =========================================================

genre_long = (
    games_with_genre
    .explode("genres_filtered")
    .rename(columns={"genres_filtered": "genre"})
    .reset_index(drop=True)
)

print("explode 후 행 수:", len(genre_long))
print("장르별 게임 수:")
print(genre_long["genre"].value_counts())


# =========================================================
# 08. 등급 라벨 한글화
# =========================================================

if "performance_grade" in genre_long.columns:
    genre_long["performance_grade_label"] = genre_long["performance_grade"].map(GRADE_LABEL)

if "scale_grade" in genre_long.columns:
    genre_long["scale_grade_label"] = genre_long["scale_grade"].map(GRADE_LABEL)

if "satisfaction_grade" in genre_long.columns:
    genre_long["satisfaction_grade_label"] = genre_long["satisfaction_grade"].map(GRADE_LABEL)


# =========================================================
# 09. Tableau에서 쓸 컬럼만 선택
# =========================================================

keep_cols = [
    "appid",
    "name",
    "genre",
    "price",
    "positive",
    "negative",
    "total_reviews",
    "positive_rate",
    "scale_grade",
    "scale_grade_label",
    "satisfaction_grade",
    "satisfaction_grade_label",
    "performance_grade",
    "performance_grade_label",
    "tags"
]

keep_cols = [col for col in keep_cols if col in genre_long.columns]

tableau_genre_long = genre_long[keep_cols].copy()


# =========================================================
# 10. 저장
# =========================================================

tableau_genre_long.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print("저장 완료:", SAVE_PATH)
print("Tableau용 데이터 크기:", tableau_genre_long.shape)
display(tableau_genre_long.head())

원본 데이터 크기: (8730, 25)
['appid', 'positive', 'negative', 'price', 'genres', 'total_reviews', 'name', 'developers', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher', 'tags', 'positive_rate', 'scale_grade', 'satisfaction_grade', 'performance_grade', 'grade_label']
변환 확인:


,appid,name,genres,genres_list
0,226620,Desktop Dungeons,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...","[Adventure, Casual, Indie, RPG, Strategy]"
1,251570,7 Days to Die,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...","[Action, Adventure, Indie, RPG, Simulation, St..."
2,252190,Defender's Quest 2: Mists of Ruin,"['Indie', 'RPG', 'Strategy']","[Indie, RPG, Strategy]"
3,269770,Secrets of Grindea,"['Action', 'Adventure', 'Indie', 'RPG']","[Action, Adventure, Indie, RPG]"
4,276870,Dwelvers,"['Indie', 'Simulation', 'Strategy']","[Indie, Simulation, Strategy]"


대상 장르가 있는 원본 게임 수: 8730
explode 후 행 수: 19009
장르별 게임 수:
genre
Adventure     4496
Casual        3870
Action        3846
Simulation    2308
RPG           1967
Strategy      1916
Sports         323
Racing         283
Name: count, dtype: int64
저장 완료: ../../../../data/preprocessed/tableau_genre_long.csv
Tableau용 데이터 크기: (19009, 15)


,appid,name,genre,price,positive,negative,total_reviews,positive_rate,scale_grade,scale_grade_label,satisfaction_grade,satisfaction_grade_label,performance_grade,performance_grade_label,tags
0,226620,Desktop Dungeons,Adventure,14.99,1912,364,2276,84.007030,high,높음,high,높음,high_high,NaN,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
1,226620,Desktop Dungeons,Casual,14.99,1912,364,2276,84.007030,high,높음,high,높음,high_high,NaN,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
2,226620,Desktop Dungeons,RPG,14.99,1912,364,2276,84.007030,high,높음,high,높음,high_high,NaN,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
3,226620,Desktop Dungeons,Strategy,14.99,1912,364,2276,84.007030,high,높음,high,높음,high_high,NaN,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
4,251570,7 Days to Die,Action,44.99,327889,42157,370046,88.607633,high,높음,high,높음,high_high,NaN,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""..."


In [3]:
tableau_genre_long['appid'].nunique()


8730